In [1]:

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Same documents from Notebook 01
documents = [
    "JES2 spool utilization is increasing due to retained job output.",
    "High JES2 spool usage can delay batch job processing.",
    "Batch workload performance is affected by resource contention.",
    "JES2 spool cleanup helps reclaim disk capacity.",
    "CICS transaction response time is increasing.",
    "Db2 query optimization improves database performance.",
    "TCP/IP network configuration on z/OS.",
    "Mainframe CPU utilization is high during peak workload.",
    "Storage shortages can cause batch job failures.",
    "Use SDSF to monitor JES2 spool and job output."
]

# Generate normalized embeddings
embeddings = model.encode(
    documents,
    normalize_embeddings=True
).astype("float32")

print("Embedding shape:", embeddings.shape)


/Users/chinmaychandrashekharhegde/WorkSpace/personal_repos/rag-engineering-lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1021 11:41:56.084000 18383 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7823.21it/s]


Embedding shape: (10, 384)


In [2]:

vectors = embeddings.astype("float32")

dimension = vectors.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(vectors)

print("Dimensions:", dimension)
print("Indexed vectors:", index.ntotal)


Dimensions: 384
Indexed vectors: 10


In [3]:

query = "Why are my batch jobs running slowly?"

query_vector = model.encode(
    [query],
    normalize_embeddings=True
).astype("float32")

scores, indices = index.search(query_vector, k=3)

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]), start=1
):
    print(
        f"{rank}. D{idx+1} | "
        f"Score={score:.4f} | {documents[idx]}"
    )


1. D3 | Score=0.6692 | Batch workload performance is affected by resource contention.
2. D9 | Score=0.5858 | Storage shortages can cause batch job failures.
3. D2 | Score=0.5803 | High JES2 spool usage can delay batch job processing.
